In [ ]:
import os
import gzip
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

In [ ]:
# Supress copy warning.
pd.options.mode.chained_assignment = None

In [ ]:
# Release of interest
rel = 12

In [ ]:
wd = '/YOUR/WORKING/DIR'

### Individual Sample Exploration

In [ ]:
# Keep EUR samples that passed QC in both NBA and WGS
concordance_dir = f'{wd}/working/R{rel}_prep/concordance'

master = pd.read_csv(f'{wd}/release/CURRENT_master_key_nba_wgs_04_06_2026.csv', low_memory = False)
master = master[(master.nba == 1) & (master.wgs == 1) & (master.nba_prune_reason.isnull()) & (master.wgs_prune_reason.isnull())]
master = master[(master.nba_label == 'EUR') & (master.wgs_label == 'EUR')]
display(master.ploidy_estimation.value_counts())

# The sample lists below were drawn once and saved so the comparison stays reproducible
# female_samples = master[master.ploidy_estimation == 'XX'].sample(n=500, random_state=42)
# male_samples = master[master.ploidy_estimation == 'XY'].sample(n=500, random_state=42)
# female_samples[['FID', 'GP2ID']].to_csv(f'{concordance_dir}/keep_female_samples.txt', index = False, header = None, sep = '\t')
# male_samples[['FID', 'GP2ID']].to_csv(f'{concordance_dir}/keep_male_samples.txt', index = False, header = None, sep = '\t')

female_samples = pd.read_csv(f'{concordance_dir}/keep_female_samples.txt', names = ['FID', 'GP2ID'], sep = '\t')
male_samples = pd.read_csv(f'{concordance_dir}/keep_male_samples.txt', names = ['FID', 'GP2ID'], sep = '\t')

female_samples = master[master.GP2ID.isin(female_samples.GP2ID)]
male_samples = master[master.GP2ID.isin(male_samples.GP2ID)]

display(female_samples.nba_prune_reason.value_counts())
display(female_samples.wgs_prune_reason.value_counts())
display(female_samples.ploidy_estimation.value_counts())

display(male_samples.nba_prune_reason.value_counts())
display(male_samples.wgs_prune_reason.value_counts())
display(male_samples.ploidy_estimation.value_counts())

In [ ]:
# Clinical - double check
clinical = pd.read_csv(f'{wd}/genotools/clinical/GP2_master_key_MAY_2026.txt', sep = '\t', low_memory = False)

female = clinical[clinical.GP2sampleID.isin(female_samples.GP2sampleID)]
male = clinical[clinical.GP2sampleID.isin(male_samples.GP2sampleID)]

display(female.sex_for_qc.value_counts())
display(male.sex_for_qc.value_counts())

In [ ]:
# Check new SNP metrics
chrom = 'X'
sample = male.IID.iloc[1]
barcode = sample.split('_')[0]

snp_metrics = pd.read_parquet(f'{wd}/snp_metrics/metrics/{barcode}/{sample}/chromosome={chrom}')
display(snp_metrics)

In [ ]:
# Double check how many SNPs in PAR and non-PAR regions in SNP metrics
# PAR1: 10001-2781479, PAR2: 155701383-156030895
nba_metadata = pd.read_parquet(f'{wd}/snp_metrics/NBA_metadata/CHROM={chrom}')
par1_ids = nba_metadata[nba_metadata["POS"].between(10001, 2781479)]
par2_ids = nba_metadata[nba_metadata["POS"].between(155701383, 156030895)]
par_ids = nba_metadata[
    nba_metadata["POS"].between(10001, 2781479)
    | nba_metadata["POS"].between(155701383, 156030895)
]
non_par_ids = nba_metadata[(~nba_metadata.snpID.isin(par1_ids.snpID)) & (~nba_metadata.snpID.isin(par2_ids.snpID))]

# R12 variant IDs are chr:pos:ref:alt
for ids in (par1_ids, par2_ids, par_ids, non_par_ids):
    ids['keep_snpID'] = 'chrX:' + ids.POS.astype(str) + ':' + ids.REF.astype(str) + ':' + ids.ALT.astype(str)

par = np.concatenate([par1_ids.snpID.unique(), par2_ids.snpID.unique()], axis=0)
non_par = non_par_ids.snpID.unique()

print(len(par))
print(len(non_par))

In [ ]:
# Joint Called WGS
joint_call_path = f'{wd}/release/dragen/r12_release/plink/EUR/chrX'
wgs = pd.read_csv(f'{joint_call_path}.pvar', names = ['#CHROM', 'POS', 'ID', 'REF', 'ALT', 'QUAL', 'FILTER', 'INFO'], sep = '\s+', comment = '#')
display(wgs)

In [ ]:
# New SNP metrics with WGS variants
keep_vars = non_par_ids[non_par_ids.keep_snpID.isin(wgs.ID)]
metrics_sample = snp_metrics[['snpID', 'LRR', 'R', 'THETA', 'GT']][snp_metrics.snpID.isin(keep_vars.snpID)]
display(metrics_sample.GT.value_counts(dropna = False))
print(len(metrics_sample))

### Gather SNP metrics

In [ ]:
sex = 'female'
region = 'PAR'
chrom = "X"

samples = female.IID.unique() if sex == 'female' else male.IID.unique()
print(len(samples))

In [ ]:
# Concatenate the per-sample SNP metrics for the region of interest
out_path = f'{concordance_dir}/{sex}_{region}_nba_new.parquet'
writer = None

for sample in samples:
    barcode = sample.split("_")[0]
    path = f'{wd}/snp_metrics/metrics/{barcode}/{sample}/chromosome={chrom}'

    if os.path.isdir(path):
        snp_metrics = pd.read_parquet(path, columns=["snpID", "GT"])
        if region == 'PAR':
            snp_metrics = snp_metrics[snp_metrics["snpID"].isin(par)] # filter SNPs
        elif region == 'non_PAR':
            snp_metrics = snp_metrics[snp_metrics["snpID"].isin(non_par)] # filter SNPs

        snp_metrics["IID"] = sample

        table = pa.Table.from_pandas(snp_metrics, preserve_index=False)

        if writer is None:
            writer = pq.ParquetWriter(out_path, table.schema)
        writer.write_table(table)

if writer is not None:
    writer.close()

In [ ]:
# Explore new concatenated SNP metrics output - All similar to individual level (just more variants)
nba = pd.read_parquet(f'{concordance_dir}/{sex}_{region}_nba_new.parquet')

nba_sample = nba[nba.IID == sample]
display(nba_sample.GT.value_counts(dropna = False))
print(len(nba_sample))
print(len(nba.IID.unique()))

### Gather Joint Call Results

In [ ]:
# Keep variants file, written once per region
keep_variants = f'{concordance_dir}/{region}_keep_variants.txt'

if not os.path.exists(keep_variants):
    with open(keep_variants, 'w') as f:
        if region == 'PAR':
            for snpID in par1_ids.keep_snpID:
                f.write(f"{snpID}\n")
            for snpID in par2_ids.keep_snpID:
                f.write(f"{snpID}\n")
        elif region == 'non_PAR':
            for snpID in non_par_ids.keep_snpID:
                f.write(f"{snpID}\n")

### VCF Method for Joint Call

In [ ]:
# Subset samples + variants out of the joint call and export to VCF
raw_prefix = f'/dev/shm/hold_genotypes/{sex}_chrX_wgs_subset_EUR_{region}'

! mkdir -p /dev/shm/hold_genotypes
! plink2 --pfile {joint_call_path} \
    --keep {concordance_dir}/keep_{sex}_samples.txt \
    --extract {keep_variants} \
    --export vcf 'bgz' \
    --out {raw_prefix}

! gzip -cd {raw_prefix}.vcf.gz | tail

In [ ]:
vcf_path = f'{raw_prefix}.vcf.gz'
out_path = f'{concordance_dir}/{sex}_{region}_joint_call.parquet'

# Parse VCF header to get column names (sample IDs = GP2IDs)
with gzip.open(vcf_path, 'rt') as f:
    for line in f:
        if line.startswith('#CHROM'):
            cols = line.strip().lstrip('#').split('\t')
            break

sample_cols = cols[9:]
vcf = pd.read_csv(vcf_path, sep='\t', comment='#', header=None, names=cols, compression='gzip',
                  usecols=['ID'] + sample_cols)
vcf.rename(columns={'ID': 'snpID'}, inplace=True)

# Melt to long format: one row per (snpID, sample)
vcf_long = vcf.melt(id_vars='snpID', var_name='GP2ID', value_name='_geno')

# Extract GT (first FORMAT field) and map to BB/AB/AA/NC
vcf_long['GT'] = vcf_long['_geno'].str.split(':').str[0]
gt_map = {
    '0/0': 'BB', '0|0': 'BB', '0': 'BB',
    '0/1': 'AB', '1/0': 'AB', '0|1': 'AB', '1|0': 'AB',
    '1/1': 'AA', '1|1': 'AA', '1': 'AA',
    './.': 'NC', '.|.': 'NC', '.':  'NC',
}
vcf_long['GT'] = vcf_long['GT'].map(gt_map).fillna('NC')
vcf_long = vcf_long[['snpID', 'GT', 'GP2ID']]

vcf_long.to_parquet(out_path, index=False)
display(vcf_long.GT.value_counts(dropna=False))
display(vcf_long.head())

### PLINK Method for Joint Call

In [ ]:
out_path = f'{concordance_dir}/{sex}_{region}_joint_call_PLINK.parquet'

# Subset samples + variants and export additive genotypes straight from the PLINK2 fileset
# --export A writes a .raw table: rows = samples, cols = variants, values = ALT allele count (0/1/2, NA = missing)
! plink2 --pfile {joint_call_path} \
    --keep {concordance_dir}/keep_{sex}_samples.txt \
    --extract {keep_variants} \
    --export A \
    --out {raw_prefix}

# Read the .raw genotype table
raw = pd.read_csv(f'{raw_prefix}.raw', sep='\t')

# First 6 cols are PLINK metadata (FID/IID/PAT/MAT/SEX/PHENOTYPE); the rest are variants
id_cols = ['FID', 'IID', 'PAT', 'MAT', 'SEX', 'PHENOTYPE']
geno_cols = [c for c in raw.columns if c not in id_cols]

# Melt to long format: one row per (snpID, sample). IID = GP2ID
geno_long = raw.melt(id_vars='IID', value_vars=geno_cols, var_name='snpID', value_name='_dosage')
geno_long.rename(columns={'IID': 'GP2ID'}, inplace=True)

# plink2 appends '_<counted_allele>' to each variant ID -> strip it to recover snpID
geno_long['snpID'] = geno_long['snpID'].str.rsplit('_', n=1).str[0]

# Map ALT-allele dosage to BB/AB/AA/NC (0 = hom REF, 1 = het, 2 = hom ALT, NA = no call)
dosage_map = {0: 'BB', 1: 'AB', 2: 'AA', -9: 'NC', '-9': 'NC', '.': 'NC'}
geno_long['GT'] = geno_long['_dosage'].map(dosage_map).fillna('NC')
geno_long = geno_long[['snpID', 'GT', 'GP2ID']]

geno_long.to_parquet(out_path, index=False)
display(geno_long.GT.value_counts(dropna=False))
display(geno_long.head())

### Calculate Concordance

In [ ]:
# NOTE: the two joint-call exports count opposite alleles, so the NBA GT codes are
# mapped differently depending on which one is being compared
method = 'PLINK' # VCF or PLINK

nba = pd.read_parquet(f'{concordance_dir}/{sex}_{region}_nba_new.parquet')
if method == 'VCF':
    nba.GT = nba.GT.map({0: 'BB', 1: 'AB', 2: 'AA', -9: 'NC'})
else:
    nba.GT = nba.GT.map({2: 'BB', 1: 'AB', 0: 'AA', -9: 'NC'})

if region == 'non_PAR':
    nba = nba.merge(non_par_ids[['snpID', 'keep_snpID']], on = 'snpID', how = 'inner')
elif region == 'PAR':
    nba = nba.merge(par_ids[['snpID', 'keep_snpID']], on = 'snpID', how = 'inner')
nba.drop(columns = 'snpID', inplace = True)
nba.rename(columns = {'keep_snpID': 'snpID'}, inplace = True)

if method == 'VCF':
    wgs = pd.read_parquet(f'{concordance_dir}/{sex}_{region}_joint_call.parquet')
    # VCF sample names repeat the GP2ID, so keep only the first half
    wgs['GP2ID'] = wgs['GP2ID'].str.split('_').apply(lambda parts: '_'.join(parts[:len(parts) // 2]))
else:
    wgs = pd.read_parquet(f'{concordance_dir}/{sex}_{region}_joint_call_PLINK.parquet')

if sex == 'male':
    wgs = wgs.merge(male[['GP2ID', 'IID']], on = 'GP2ID', how = 'inner')
elif sex == 'female':
    wgs = wgs.merge(female[['GP2ID', 'IID']], on = 'GP2ID', how = 'inner')
wgs = wgs[['snpID', 'GT', 'IID']]

display(nba)
print(len(nba.IID.unique()))
display(wgs)
print(len(wgs.IID.unique()))

In [ ]:
# Calculate concordance -- merge on sample ID and variant ID
merged = (nba.merge(wgs, on=["IID", "snpID"], suffixes=("_nba", "_wgs")))
merged["concordant"] = merged["GT_nba"] == merged["GT_wgs"]

print("Overall Concordance:", merged["concordant"].mean())
print("Overall Concordance with no NC in WGS:",
      merged[merged["GT_wgs"] != "NC"]["concordant"].mean())
print("Overall Concordance with no NC in WGS or NBA:",
      merged[(merged["GT_nba"] != "NC") & (merged["GT_wgs"] != "NC")]["concordant"].mean())

In [ ]:
# Genotype breakdown of the discordant calls
display(merged.GT_wgs.value_counts())
display(merged[~merged.concordant].GT_nba.value_counts())
display(merged[~merged.concordant].GT_wgs.value_counts())

print(clinical[clinical.IID.isin(merged.IID)].sex_for_qc.value_counts())
print(len(merged.IID.unique()))

In [ ]:
# Share of discordant calls carried by each genotype
nba_disc = merged[~merged.concordant].GT_nba.value_counts().reset_index()
nba_disc['disc_percent'] = (nba_disc['count'] / len(merged[~merged.concordant])) * 100
nba_disc['percent'] = (nba_disc['count'] / len(merged)) * 100
display(nba_disc)

wgs_disc = merged[~merged.concordant].GT_wgs.value_counts().reset_index()
wgs_disc['disc_percent'] = (wgs_disc['count'] / len(merged[~merged.concordant])) * 100
wgs_disc['percent'] = (wgs_disc['count'] / len(merged)) * 100
display(wgs_disc)

In [ ]:
# Heterozygous calls are the ones to watch in males
display(merged[merged.GT_nba == 'AB'])
display(merged[merged.GT_wgs == 'AB'])
display(merged[(merged.GT_nba == 'NC') & (merged.GT_wgs == 'NC')])